# TravelTide: Customer-Level Feature Engineering

This notebook builds one record per customer from raw user, session, flight, and hotel exports. It uses noncanceled bookings for booking-value and travel features while retaining session engagement and cancellation measures.

## Data and outputs
Raw inputs: `data/raw`. Derived outputs: `data/own_processed`. Raw inputs are not overwritten; reruns replace the derived files and build summary.

## Hotel-night methodology
Hotel nights are calculated using check-out date minus check-in date, rather than elapsed whole 24-hour periods. A Friday-afternoon arrival and Sunday-morning departure represent two nights. Original values are retained. Same-day stays, reversed timestamps, and missing dates are flagged as unresolved; their booking counts are retained but duration-dependent spending is not estimated.

## Interpretation
The reference date is the latest observed session-start calendar date. Features use the full exported history of the selected cohort. Booking expenditure is an estimate, not platform revenue or profit. The fare is assumed to cover the whole flight booking; hotel price is assumed to be per room-night. This notebook does not fit clustering models or assign perks.


In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

# Run from the project root or its notebooks folder.
PROJECT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT is None:
    raise FileNotFoundError("Open this notebook from the traveltide_submission project.")
DATA = PROJECT / "data"
RAW = DATA / "raw"
OUT = DATA / "own_processed"
paths = {name: RAW / f"{name}.csv" for name in ["users", "sessions", "flights", "hotels"]}
missing = [str(p) for p in paths.values() if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing files:\n" + "\n".join(missing))
tables = {name: pd.read_csv(path, encoding="utf-8-sig", low_memory=False,
          dtype={"user_id":"string", "trip_id":"string", "session_id":"string"})
          for name,path in paths.items()}
u,s,f,h = [tables[name].copy() for name in ["users","sessions","flights","hotels"]]

def dt(series):
    parsed = pd.to_datetime(series, format="mixed", errors="coerce")
    if (series.notna() & parsed.isna()).any():
        raise ValueError(f"Unparseable nonmissing date in {series.name}; no outputs written.")
    return parsed

def flag(series):
    result = series.astype("string").str.strip().str.lower().map(
        {"true":True,"false":False,"1":True,"0":False,"1.0":True,"0.0":False})
    if result.isna().any():
        raise ValueError(f"Missing/unknown boolean in {series.name}; no outputs written.")
    return result.astype(bool)

def num(series):
    return pd.to_numeric(series,errors="coerce")

for name,df,key in [("users",u,"user_id"),("sessions",s,"session_id"),
                    ("flights",f,"trip_id"),("hotels",h,"trip_id")]:
    if df[key].isna().any() or not df[key].is_unique:
        raise ValueError(f"{name}.{key} must be complete and unique; no outputs written.")
if not s["user_id"].isin(u["user_id"]).all():
    raise ValueError("Session user not found in users.csv; no outputs written.")
for c in ["cancellation","flight_booked","hotel_booked","flight_discount","hotel_discount"]:
    s[c]=flag(s[c])
s["session_start"]=dt(s["session_start"])
s["session_end"]=dt(s["session_end"])
if s["session_start"].isna().any():
    raise ValueError("Missing session start; no outputs written.")
REFERENCE = s["session_start"].max().normalize()
ELIGIBILITY_START = pd.Timestamp("2023-01-05")
# Full exported history is used for features; recent-session filter defined eligibility.
canceled_ids = set(s.loc[s["cancellation"],"trip_id"].dropna())
original = s.loc[~s["cancellation"] & s["trip_id"].notna()
                 & (s["flight_booked"] | s["hotel_booked"])].copy()
if not original["trip_id"].is_unique:
    raise ValueError("Multiple original booking events for a trip; no outputs written.")
if not f["trip_id"].isin(original["trip_id"]).all() or not h["trip_id"].isin(original["trip_id"]).all():
    raise ValueError("Product record lacks original booking event; no outputs written.")
bookings = original.loc[~original["trip_id"].isin(canceled_ids)].copy()
bookings["booking_date"] = bookings["session_start"].dt.normalize()
bookings["product_matched_discount"] = ((bookings["flight_booked"] & bookings["flight_discount"])
                                       | (bookings["hotel_booked"] & bookings["hotel_discount"]))
# Treat missing amount as zero ONLY with an explicit false product-discount flag.
for product in ["flight","hotel"]:
    amount=product+"_discount_amount"; indicator=product+"_discount"
    values=num(bookings[amount])
    no_discount=~bookings[indicator]
    consistent_no_discount=no_discount & (values.isna() | values.eq(0))
    valid_discount=bookings[indicator] & values.gt(0) & values.le(1)
    bookings[product+"_discount_fraction_used"]=values.where(valid_discount)
    bookings.loc[consistent_no_discount,product+"_discount_fraction_used"]=0.0

mapping_cols=["trip_id","user_id","session_start","booking_date","flight_booked","hotel_booked",
              "flight_discount","hotel_discount","flight_discount_fraction_used","hotel_discount_fraction_used"]
flight=f.loc[~f["trip_id"].isin(canceled_ids)].merge(bookings[mapping_cols],on="trip_id",how="left",validate="one_to_one")
hotel=h.loc[~h["trip_id"].isin(canceled_ids)].merge(bookings[mapping_cols],on="trip_id",how="left",validate="one_to_one")
if flight["user_id"].isna().any() or hotel["user_id"].isna().any():
    raise ValueError("Missing surviving booking mapping; no outputs written.")
if not flight["flight_booked"].all() or not hotel["hotel_booked"].all():
    raise ValueError("Product record disagrees with booking flag; no outputs written.")

flight["departure_time"]=dt(flight["departure_time"])
flight["return_time"]=dt(flight["return_time"])
flight["flight_lead_days"]=(flight["departure_time"]-flight["session_start"]).dt.total_seconds()/86400
flight["flight_spend_estimate"]=(num(flight["base_fare_usd"])*(1-flight["flight_discount_fraction_used"]))
flight.loc[num(flight["base_fare_usd"]).lt(0),"flight_spend_estimate"]=np.nan
# base_fare_usd is treated as the whole booking fare, NOT a per-seat price.
flight["base_price_per_seat"]=num(flight["base_fare_usd"])/num(flight["seats"]).where(num(flight["seats"]).gt(0))

hotel["check_in_time"]=dt(hotel["check_in_time"])
hotel["check_out_time"]=dt(hotel["check_out_time"])
hotel["nights_original"]=num(hotel["nights"])
hotel["calendar_nights"]=(hotel["check_out_time"].dt.normalize()-hotel["check_in_time"].dt.normalize()).dt.days
chronological=hotel["check_out_time"].gt(hotel["check_in_time"])
valid_nights=chronological & hotel["calendar_nights"].gt(0)
hotel["nights_clean"]=hotel["calendar_nights"].where(valid_nights)
hotel["nights_unresolved"]=~valid_nights
hotel["nights_corrected"]=valid_nights & hotel["nights_original"].ne(hotel["nights_clean"])
hotel["hotel_spend_estimate"]=(hotel["nights_clean"]*num(hotel["rooms"])
    *num(hotel["hotel_per_room_usd"])*(1-hotel["hotel_discount_fraction_used"]))
hotel.loc[num(hotel["rooms"]).le(0) | num(hotel["hotel_per_room_usd"]).lt(0),"hotel_spend_estimate"]=np.nan
hotel["hotel_lead_days"]=(hotel["check_in_time"]-hotel["session_start"]).dt.total_seconds()/86400
hotel["fri_sat_checkin"]=hotel["check_in_time"].dt.weekday.isin([4,5])
hotel["short_weekend_stay"]=(hotel["fri_sat_checkin"] & hotel["nights_clean"].le(3)).astype(float)
hotel.loc[hotel["nights_unresolved"],"short_weekend_stay"]=np.nan

features=u.copy().set_index("user_id")
u_signup=dt(features["sign_up_date"]); birth=dt(features["birthdate"])
features["signup_tenure_days"]=(REFERENCE-u_signup.dt.normalize()).dt.days
birthday_pending=(birth.dt.month.gt(REFERENCE.month)
    | (birth.dt.month.eq(REFERENCE.month)&birth.dt.day.gt(REFERENCE.day)))
features["age_at_reference"]=REFERENCE.year-birth.dt.year-birthday_pending.astype(int)
features["num_sessions"]=s.groupby("user_id")["session_id"].nunique().reindex(features.index,fill_value=0)
features["num_recent_sessions"]=s.loc[s["session_start"].ge(ELIGIBILITY_START)].groupby("user_id")["session_id"].nunique().reindex(features.index,fill_value=0)
features["session_recency_days"]=(REFERENCE-s.groupby("user_id")["session_start"].max().dt.normalize()).dt.days
features["average_session_seconds"]=((s["session_end"]-s["session_start"]).dt.total_seconds()).groupby(s["user_id"]).mean()
features["average_page_clicks"]=num(s["page_clicks"]).groupby(s["user_id"]).mean()
features["num_original_booked_trips"]=original.groupby("user_id")["trip_id"].nunique().reindex(features.index,fill_value=0)
features["num_canceled_trips"]=original.loc[original["trip_id"].isin(canceled_ids)].groupby("user_id")["trip_id"].nunique().reindex(features.index,fill_value=0)
features["valid_trip_count"]=bookings.groupby("user_id")["trip_id"].nunique().reindex(features.index,fill_value=0)
features["cancellation_rate"]=features["num_canceled_trips"]/features["num_original_booked_trips"].replace(0,np.nan)
# Conversion denominator excludes cancellation visits, but includes browsing.
features["noncancellation_session_count"]=s.loc[~s["cancellation"]].groupby("user_id")["session_id"].nunique().reindex(features.index,fill_value=0)
features["net_booking_conversion"]=features["valid_trip_count"]/features["noncancellation_session_count"].replace(0,np.nan)
first=bookings.groupby("user_id")["booking_date"].min(); last=bookings.groupby("user_id")["booking_date"].max()
features["first_valid_booking_date"]=first; features["last_valid_booking_date"]=last
features["booking_recency_days"]=(REFERENCE-last).dt.days
features["booking_relationship_length_days"]=(last-first).dt.days
features["discounted_valid_trip_count"]=bookings.loc[bookings["product_matched_discount"]].groupby("user_id")["trip_id"].nunique().reindex(features.index,fill_value=0)
features["discounted_valid_trip_rate"]=features["discounted_valid_trip_count"]/features["valid_trip_count"].replace(0,np.nan)
features["booking_status"]=np.select(
    [features["valid_trip_count"].gt(0),features["num_original_booked_trips"].gt(0)],
    ["Has noncanceled bookings","Only canceled bookings"],default="No observed bookings")

def product_features(df,product,spend_column):
    group=df.groupby("user_id")
    counts=group["trip_id"].nunique().reindex(features.index,fill_value=0)
    features["num_"+product+"_trips"]=counts
    discount_count=df.loc[df[product+"_discount"]].groupby("user_id")["trip_id"].nunique().reindex(features.index,fill_value=0)
    features[product+"_discounted_booking_count"]=discount_count
    features[product+"_discounted_booking_rate"]=discount_count/counts.replace(0,np.nan)
    unknown=group[spend_column].apply(lambda x:int(x.isna().sum())).reindex(features.index,fill_value=0)
    features[product+"_spend_unknown_booking_count"]=unknown
    subtotal=group[spend_column].sum(min_count=1).reindex(features.index)
    subtotal.loc[counts.eq(0)]=0.0
    # Complete total remains unknown if ANY component booking is unresolved.
    features["total_"+product+"_spend_estimate"]=subtotal.mask(unknown.gt(0))

product_features(flight,"flight","flight_spend_estimate")
product_features(hotel,"hotel","hotel_spend_estimate")
features["total_booking_spend_estimate"]=features["total_flight_spend_estimate"]+features["total_hotel_spend_estimate"]
features["spend_complete"]=features["total_booking_spend_estimate"].notna()
features["average_seats"]=num(flight["seats"]).groupby(flight["user_id"]).mean()
features["average_checked_bags"]=num(flight["checked_bags"]).groupby(flight["user_id"]).mean()
features["average_base_price_per_seat"]=flight.groupby("user_id")["base_price_per_seat"].mean()
features["average_flight_lead_days"]=flight.groupby("user_id")["flight_lead_days"].mean()
features["average_hotel_lead_days"]=hotel.groupby("user_id")["hotel_lead_days"].mean()
features["hotel_nights_unresolved_booking_count"]=hotel.groupby("user_id")["nights_unresolved"].sum().reindex(features.index,fill_value=0)
features["hotel_nights_corrected_booking_count"]=hotel.groupby("user_id")["nights_corrected"].sum().reindex(features.index,fill_value=0)
features["average_known_hotel_nights"]=hotel.groupby("user_id")["nights_clean"].mean()
features["known_hotel_nights_booking_count"]=hotel.groupby("user_id")["nights_clean"].count().reindex(features.index,fill_value=0)
features["fri_sat_hotel_checkin_share"]=hotel.groupby("user_id")["fri_sat_checkin"].mean()
features["short_weekend_stay_share_known_duration"]=hotel.groupby("user_id")["short_weekend_stay"].mean()
features["flight_hotel_combined_trip_count"]=bookings.loc[bookings["flight_booked"]&bookings["hotel_booked"]].groupby("user_id")["trip_id"].nunique().reindex(features.index,fill_value=0)
features["hotel_on_flight_trip_share"]=features["flight_hotel_combined_trip_count"]/features["num_flight_trips"].replace(0,np.nan)
features["analysis_reference_date"]=REFERENCE.date().isoformat()
assert features.index.is_unique and len(features)==len(u)
if np.isinf(features.select_dtypes(include="number").to_numpy(dtype=float)).any():
    raise ValueError("Infinite feature values; no outputs written.")
OUT.mkdir(parents=True,exist_ok=True)
features.reset_index().to_csv(OUT/"user_features_own.csv",index=False)
hotel.to_csv(OUT/"hotels_own_processed.csv",index=False)
flight.to_csv(OUT/"flights_own_processed.csv",index=False)
summary=[
    "TRAVELTIDE — CUSTOMER FEATURE BUILD SUMMARY",
    f"Generated: {datetime.now().astimezone().isoformat(timespec='seconds')}",
    f"Inputs: {RAW}",f"Outputs: {OUT}",f"Reference date: {REFERENCE.date()}",
    "Scope: selected customers, full exported booking/session history.",
    "Raw CSVs and instructor files were not modified. Reruns replace only these own outputs.",
    "", "HOTEL-NIGHT METHODOLOGY:",
    "Hotel nights count calendar dates, not completed 24-hour periods. Friday afternoon to Sunday morning is two nights even though fewer than 48 hours elapsed.",
    "Original nights are preserved. Positive calendar-date differences are used only when check-out is later than check-in.",
    "Same-day, reversed, or missing dates are flagged; no night count is invented. Their bookings remain in counts, but their hotel spending is unknown.",
    "A customer's full spending total is unknown when any surviving booking has unknown spending; this is not silently replaced with zero.",
    "Average known hotel nights uses only resolved stays, with the supporting count provided.",
    "", "OTHER DEFINITIONS:",
    "Age and tenure use the fixed reference date. Booking recency uses noncanceled booking dates, not future departure dates.",
    "Length = days between first and last observed noncanceled bookings; signup tenure is a separate feature. Single-booking Length is zero; no-booker Length/Recency is missing.",
    "Frequency = distinct noncanceled trips in the full export. Monetary candidate = estimated flight plus hotel booking expenditure, not OTA revenue or profit.",
    "Fare assumption: base_fare_usd is a whole-booking fare. Hotel-price assumption: hotel_per_room_usd is price per room-night. Discount amounts are treated as fractions.",
    "Discount rates count only the booked product; missing amounts become zero only with an explicit false discount flag.",
    "Net booking conversion = noncanceled trips / noncancellation sessions; browsing remains in the denominator.",
    "Weekend measures are named as Friday/Saturday check-in share and short stays of at most three known nights; neither proves leisure travel.",
    "No clustering, final perk assignment, outlier deletion, or model imputation was performed.",
    "", "BUILD RESULTS:",f"Customers: {len(features)}",
    f"Noncanceled flight / hotel records: {len(flight)} / {len(hotel)}",
    f"Hotel records with corrected nights: {int(hotel['nights_corrected'].sum())}",
    f"Hotel records with unresolved nights: {int(hotel['nights_unresolved'].sum())}",
    f"Customers with incomplete total spending: {int((~features['spend_complete']).sum())}",
    f"All hotel check-outs at 11:00:00: {bool((hotel['check_out_time'].notna() & hotel['check_out_time'].eq(hotel['check_out_time'].dt.normalize()+pd.Timedelta(hours=11))).all())}",
    "Booking status counts: "+str(features['booking_status'].value_counts().to_dict()),
    "", "OUTPUT COLUMNS:",*features.reset_index().columns.tolist(),
]
(PROJECT/"feature_build_summary.txt").write_text("\n".join(summary),encoding="utf-8")
print("Done. Raw files unchanged.")
print(f"Customer features: {OUT / 'user_features_own.csv'}")
print(f"Build summary: {PROJECT / 'feature_build_summary.txt'}")
